In [ ]:
from dataclasses import dataclass
from pathlib import Path
import csv
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

In [ ]:
# Use relative paths so the notebook can be moved to another machine.
MANIFEST_PATH = Path('data/manifest.csv')
FEATURE_OUTPUT = Path('outputs/dinov2_features.npz')
MODEL_SOURCE = None  # Set to Path('models/dinov2') for an offline local repository.
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE_NAME = 'auto'

In [ ]:
MODALITIES = ('rgb', 'tir', 'dem')
CLASS_NAMES = (
    'Background',
    'Debris Accumulation',
    'Exposed Rock Mass',
    'Agricultural Encroachment',
    'Drainage Infrastructure',
)
MODEL_REPOSITORY = 'facebookresearch/dinov2'
MODEL_NAME = 'dinov2_vits14_reg'
FEATURE_DIMENSION = 384

# DINOv2 ImageNet preprocessing used by the public implementation.
NORMALIZATION_MEAN = (0.485, 0.456, 0.406)
NORMALIZATION_STD = (0.229, 0.224, 0.225)
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZATION_MEAN, NORMALIZATION_STD),
])

In [ ]:
@dataclass(frozen=True)
class Sample:
    sample_id: str
    image_paths: dict[str, Path]
    label: int


def read_manifest(path: Path) -> list[Sample]:
    required = {'sample_id', *MODALITIES, 'label'}
    samples = []
    seen_ids = set()
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        missing = required.difference(reader.fieldnames or ())
        if missing:
            raise ValueError(f'Missing manifest columns: {sorted(missing)}')
        for row in reader:
            sample_id = row['sample_id'].strip()
            if not sample_id or sample_id in seen_ids:
                raise ValueError(f'sample_id must be non-empty and unique: {sample_id!r}')
            seen_ids.add(sample_id)
            image_paths = {}
            for modality in MODALITIES:
                image_path = Path(row[modality]).expanduser()
                image_paths[modality] = image_path if image_path.is_absolute() else path.parent / image_path
            if any(not image_path.is_file() for image_path in image_paths.values()):
                raise FileNotFoundError(f'Missing image for sample {row["sample_id"]}')
            label = int(row['label'])
            if label not in range(len(CLASS_NAMES)):
                raise ValueError(f'Label must be between 0 and {len(CLASS_NAMES) - 1}')
            samples.append(Sample(sample_id, image_paths, label))
    if not samples:
        raise ValueError('The manifest contains no samples.')
    return samples


class MultimodalPatchDataset(Dataset):
    def __init__(self, samples: list[Sample]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        images = []
        for modality in MODALITIES:
            with Image.open(sample.image_paths[modality]) as image:
                images.append(IMAGE_TRANSFORM(image.convert('RGB')))
        return (sample.sample_id, sample.label, *images)

In [ ]:
samples = read_manifest(MANIFEST_PATH)
device = torch.device('cuda' if DEVICE_NAME == 'auto' and torch.cuda.is_available() else 'cpu')
if DEVICE_NAME == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA was requested but is not available.')
if DEVICE_NAME in ('cpu', 'cuda'):
    device = torch.device(DEVICE_NAME)

if MODEL_SOURCE is None:
    encoder = torch.hub.load(MODEL_REPOSITORY, MODEL_NAME)
else:
    encoder = torch.hub.load(str(MODEL_SOURCE), MODEL_NAME, source='local')
encoder = encoder.to(device).eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)

dataset = MultimodalPatchDataset(samples)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda')
print(f'Samples: {len(dataset)} | Device: {device} | Model: {MODEL_NAME}')

In [ ]:
sample_ids = []
labels = []
feature_batches = {modality: [] for modality in MODALITIES}

with torch.inference_mode():
    for batch_ids, batch_labels, rgb, tir, dem in loader:
        sample_ids.extend(batch_ids)
        labels.extend(batch_labels.numpy().tolist())
        for modality, batch in zip(MODALITIES, (rgb, tir, dem)):
            encoded = encoder(batch.to(device, non_blocking=device.type == 'cuda'))
            encoded = encoded.detach().float().cpu().numpy()
            if encoded.shape[1] != FEATURE_DIMENSION:
                raise ValueError(f'{modality} feature shape is {encoded.shape}, expected 384 columns.')
            feature_batches[modality].append(encoded)

feature_matrices = {
    modality: np.concatenate(feature_batches[modality]).astype(np.float32)
    for modality in MODALITIES
}
FEATURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    FEATURE_OUTPUT,
    sample_ids=np.asarray(sample_ids, dtype=str),
    labels=np.asarray(labels, dtype=np.int64),
    **feature_matrices,
    model_name=np.asarray(MODEL_NAME),
    feature_dimension=np.asarray(FEATURE_DIMENSION, dtype=np.int64),
)
print(f'Saved features to {FEATURE_OUTPUT}')